# Repeated Composition-Level GroupKFold: Random Forest Feature-Set Benchmark

**Purpose:** Evaluate Random Forest using the same repeated composition-level GroupKFold protocol as the main PAQFE-v2 vs Standard QML notebook, for three feature sets: physics-only, context-only, and all features.

**Protocol:** seeds 42–46 × 5 folds; composition groups built from the five normalized elemental fractions rounded to three decimals; training/test composition overlap is checked; metrics are calculated on pooled out-of-fold predictions within each repeat, then summarized across repeats.

**Important:** Use the exact same model-ready Excel workbook and keep its row order/data cleaning consistent with the QML notebook. This notebook reconstructs the folds deterministically using the same GroupKFold settings. After running it, compare its fold audit with the saved QML fold audit if available.

## 1. Install dependencies (Google Colab only, if needed)

If scikit-learn is too old to support `GroupKFold(shuffle=True, random_state=...)`, upgrade it and restart the runtime before continuing.

In [ ]:
# Uncomment in Google Colab only if needed:
# %pip install -U scikit-learn openpyxl

In [ ]:
import os
import hashlib
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

SEEDS = [42, 43, 44, 45, 46]
N_SPLITS = 5
N_ESTIMATORS = 300
TARGET = "UTS_MPa"
RANDOM_STATE_RF = 42  # fixed RF randomness; fold seed controls the data split

STANDARD_FEATURE_ORDER = [
    "xAl", "VEC", "dHmix", "dSmix", "delta",
    "Omega", "dchi", "Mean_Tm", "Temperature_C", "log10_Strain_rate"
]
PHYSICS_FEATURES = [
    "xAl", "VEC", "dHmix", "dSmix", "delta",
    "Omega", "dchi", "Mean_Tm"
]
CONTEXT_FEATURES = ["Temperature_C", "log10_Strain_rate"]
ALL_FEATURES = STANDARD_FEATURE_ORDER
COMPOSITION_COLS = ["Al_norm", "Co_norm", "Cr_norm", "Fe_norm", "Ni_norm"]

FEATURE_SETS = {
    "RF_Physics_Only": PHYSICS_FEATURES,
    "RF_Context_Only": CONTEXT_FEATURES,
    "RF_All_Features": ALL_FEATURES,
}

print("Seeds:", SEEDS)
print("Folds per repeat:", N_SPLITS)
print("RF estimators:", N_ESTIMATORS)
print("Feature-set sizes:", {k: len(v) for k, v in FEATURE_SETS.items()})

## 2. Load the same workbook and prepare composition groups

Upload the same Excel workbook used by the quantum GroupKFold notebook. The cleaning and group construction below intentionally mirror that notebook.

In [ ]:
# Google Colab upload:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file was uploaded.")
xlsx_path = next(iter(uploaded.keys()))

df = pd.read_excel(xlsx_path, sheet_name="B_Model_Ready")

required = list(dict.fromkeys(
    STANDARD_FEATURE_ORDER + [TARGET] + COMPOSITION_COLS
))
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df[required] = df[required].apply(pd.to_numeric, errors="coerce")
df = df.dropna(subset=required).reset_index(drop=True).copy()
df["Composition_Group"] = (
    df[COMPOSITION_COLS].round(3).astype(str).agg("_".join, axis=1)
)

print("Workbook:", os.path.basename(xlsx_path))
print("Cleaned observations:", len(df))
print("Unique compositions:", df["Composition_Group"].nunique())
display(df[[TARGET, "Composition_Group"]].head())

if len(df) != 49 or df["Composition_Group"].nunique() != 14:
    raise ValueError(
        "Dataset does not match expected 49 observations / 14 compositions. "
        "Check workbook version and cleaning before running."
    )

## 3. Generate and audit the shared GroupKFold splits

The split definition matches the main QML notebook: `GroupKFold(n_splits=5, shuffle=True, random_state=seed)`. The same split is reused for all three RF feature sets.

In [ ]:
groups = df["Composition_Group"].to_numpy()
X_for_split = df[STANDARD_FEATURE_ORDER]
y_for_split = df[TARGET]

split_cache = {}
split_audit = []

for seed in SEEDS:
    splitter = GroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    splits = list(splitter.split(X_for_split, y_for_split, groups))
    assert len(splits) == N_SPLITS

    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        train_groups = set(groups[train_idx])
        test_groups = set(groups[test_idx])
        overlap = train_groups.intersection(test_groups)
        assert not overlap, f"Composition leakage: seed={seed}, fold={fold}: {overlap}"

        split_cache[(seed, fold)] = (train_idx, test_idx)
        split_audit.append({
            "Seed": seed,
            "Fold": fold,
            "N_Train_Observations": len(train_idx),
            "N_Test_Observations": len(test_idx),
            "N_Train_Compositions": len(train_groups),
            "N_Test_Compositions": len(test_groups),
            "Composition_Overlap_Count": len(overlap),
            "Train_Compositions": ";".join(sorted(train_groups)),
            "Test_Compositions": ";".join(sorted(test_groups)),
        })

split_audit_df = pd.DataFrame(split_audit)
print(f"Validated {len(split_cache)} splits; no composition overlap detected.")
display(split_audit_df[[
    "Seed", "Fold", "N_Train_Observations", "N_Test_Observations",
    "N_Train_Compositions", "N_Test_Compositions", "Composition_Overlap_Count"
]])

## 4. Train RF models and collect out-of-fold predictions

Random Forest does not require feature standardization, so no scaler is fitted. Each model is trained only on the training-fold observations. `random_state` is fixed for the RF estimator so differences across repeated runs primarily reflect the changed held-out composition assignments.

In [ ]:
fold_records = []
oof_records = []

for seed in SEEDS:
    for fold in range(1, N_SPLITS + 1):
        train_idx, test_idx = split_cache[(seed, fold)]
        train_groups = set(groups[train_idx])
        test_groups = set(groups[test_idx])
        assert train_groups.isdisjoint(test_groups)

        for model_name, features in FEATURE_SETS.items():
            model = RandomForestRegressor(
                n_estimators=N_ESTIMATORS,
                random_state=RANDOM_STATE_RF,
                n_jobs=-1,
            )
            X_train = df.iloc[train_idx][features]
            X_test = df.iloc[test_idx][features]
            y_train = df.iloc[train_idx][TARGET].to_numpy(dtype=float)
            y_test = df.iloc[test_idx][TARGET].to_numpy(dtype=float)

            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            fold_records.append({
                "Model": model_name,
                "Seed": seed,
                "Fold": fold,
                "N_Train": len(train_idx),
                "N_Test": len(test_idx),
                "MAE_MPa": mean_absolute_error(y_test, y_pred),
                "RMSE_MPa": mean_squared_error(y_test, y_pred) ** 0.5,
                "R2": r2_score(y_test, y_pred) if len(y_test) > 1 else np.nan,
            })

            for local_j, row_idx in enumerate(test_idx):
                oof_records.append({
                    "Model": model_name,
                    "Seed": seed,
                    "Fold": fold,
                    "Row_Index_After_Cleaning": int(row_idx),
                    "Composition_Group": groups[row_idx],
                    "y_true_MPa": float(y_test[local_j]),
                    "y_pred_MPa": float(y_pred[local_j]),
                    "Residual_MPa": float(y_test[local_j] - y_pred[local_j]),
                })

    print(f"Completed seed {seed}")

fold_metrics = pd.DataFrame(fold_records)
oof_predictions = pd.DataFrame(oof_records)
print("Fold-metric rows:", len(fold_metrics))
print("OOF prediction rows:", len(oof_predictions))
display(fold_metrics.head())

## 5. Calculate pooled OOF metrics per repeat

For each seed, every observation receives exactly one out-of-fold prediction across the five folds. Metrics are computed on those pooled predictions, then the mean and SD are reported across the five repeats. These are descriptive summaries; the repeats reuse the same 14 compositions.

In [ ]:
pooled_records = []

for (model_name, seed), part in oof_predictions.groupby(["Model", "Seed"], sort=True):
    y_true = part["y_true_MPa"].to_numpy()
    y_pred = part["y_pred_MPa"].to_numpy()
    pooled_records.append({
        "Model": model_name,
        "Seed": int(seed),
        "N_OOF": len(part),
        "MAE_MPa": mean_absolute_error(y_true, y_pred),
        "RMSE_MPa": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
    })

pooled_repeat_metrics = pd.DataFrame(pooled_records)
summary = (
    pooled_repeat_metrics.groupby("Model")[["MAE_MPa", "RMSE_MPa", "R2"]]
    .agg(["mean", "std"])
)
print("Pooled out-of-fold metrics by repeat:")
display(pooled_repeat_metrics)
print("Summary across repeats (mean and sample SD):")
display(summary)

## 6. Optional: feature importance from each fitted fold

This is a diagnostic summary of impurity-based RF feature importance. Because features sets differ, compare within each model/feature set. For stronger interpretability, permutation importance on held-out folds is preferable; do not interpret impurity importance as causal effects.

In [ ]:
importance_records = []

for seed in SEEDS:
    for fold in range(1, N_SPLITS + 1):
        train_idx, test_idx = split_cache[(seed, fold)]
        for model_name, features in FEATURE_SETS.items():
            model = RandomForestRegressor(
                n_estimators=N_ESTIMATORS,
                random_state=RANDOM_STATE_RF,
                n_jobs=-1,
            )
            model.fit(df.iloc[train_idx][features], df.iloc[train_idx][TARGET])
            for feature, importance in zip(features, model.feature_importances_):
                importance_records.append({
                    "Model": model_name,
                    "Seed": seed,
                    "Fold": fold,
                    "Feature": feature,
                    "Importance": float(importance),
                })

importance_df = pd.DataFrame(importance_records)
importance_summary = (
    importance_df.groupby(["Model", "Feature"])["Importance"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .sort_values(["Model", "mean"], ascending=[True, False])
)
display(importance_summary)

## 7. Save outputs

The notebook saves fold metrics, pooled OOF metrics, per-observation OOF predictions, the composition split audit, and feature importance. These outputs are what we should use to update the manuscript—only after checking the split assignments against the QML notebook.

In [ ]:
OUTPUT_DIR = "RF_Repeated_GroupKFold_Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

outputs = {
    "rf_groupkfold_fold_metrics.csv": fold_metrics,
    "rf_groupkfold_oof_predictions.csv": oof_predictions,
    "rf_groupkfold_pooled_repeat_metrics.csv": pooled_repeat_metrics,
    "rf_groupkfold_split_audit.csv": split_audit_df,
    "rf_groupkfold_feature_importance.csv": importance_df,
    "rf_groupkfold_feature_importance_summary.csv": importance_summary,
}
for filename, table in outputs.items():
    table.to_csv(os.path.join(OUTPUT_DIR, filename), index=False)

# Dataset fingerprint helps identify which cleaned dataset produced these outputs.
fingerprint_cols = STANDARD_FEATURE_ORDER + [TARGET] + COMPOSITION_COLS + ["Composition_Group"]
fingerprint = hashlib.sha256(
    pd.util.hash_pandas_object(df[fingerprint_cols], index=True).values.tobytes()
).hexdigest()[:16]

with open(os.path.join(OUTPUT_DIR, "run_metadata.txt"), "w", encoding="utf-8") as f:
    f.write(f"dataset_fingerprint={fingerprint}\n")
    f.write(f"seeds={SEEDS}\n")
    f.write(f"n_splits={N_SPLITS}\n")
    f.write(f"n_estimators={N_ESTIMATORS}\n")
    f.write(f"rf_random_state={RANDOM_STATE_RF}\n")
    f.write(f"n_observations={len(df)}\n")
    f.write(f"n_compositions={df['Composition_Group'].nunique()}\n")

print("Saved files:")
for filename in outputs:
    print(os.path.join(OUTPUT_DIR, filename))
print("Dataset fingerprint:", fingerprint)

## 8. Important interpretation notes

- This is the grouped counterpart to the existing observation-level RF feature-set notebook; keep both.
- Do not assume the new RF results will reproduce the previously mentioned grouped RF summary until the experiment is run.
- Check that the split audit and cleaned row order match the main QML notebook's split audit. If they do not, compare the exact grouping and cleaning steps before interpreting model differences.
- Random Forest does not use the QML model's target scaling or optimizer; it predicts UTS directly in MPa.
- Report the grouped RF results as a classical reference, not as evidence of quantum advantage.
- The five repeated splits reuse the same compositions, so the SD across repeats is descriptive rather than an independent uncertainty estimate.